# Множественное тестирование

## Поправка Бонферрони

Почему множественное тестирование – это проблема?

При проведении нескольких статистических тестов вероятность совершения хотя бы одной ошибки первого рода увеличивается. Если мы используем стандартный уровень значимости $\alpha = 0.05$, то вероятность ложного отклонения нулевой гипотезы для одного теста составляет 5%. Однако при множественных проверках:

$$Вероятность хотя бы одной ошибки=1−(1−α)^m$$

Где:

- $\alpha$ — уровень значимости (обычно 0.05),
- $m$ — количество тестов.

Например, если провести 10 независимых тестов при $\alpha=0.05$, вероятность того, что хотя бы один из них даст ложноположительный результат:
$$1−(1−0.05)^{10} \approx 0.40$$

Таким образом, ложноположительная ошибка возрастает с увеличением числа тестов, что приводит к ложным выводам.

__Поправка Бонферрони__ — это простой и консервативный метод корректировки уровня значимости для учета множественного тестирования. Метод заключается в делении уровня значимости на количество проведенных тестов:
$$\alpha_{new} = \frac{\alpha}{m}$$
 
Где:
- $\alpha_{new}$ — скорректированный уровень значимости,
- $\alpha$ — исходный уровень значимости (например, 0.05),
- $m$ — количество тестов.

__Пример использования__:

Если мы проводим 5 тестов и используем стандартный уровень значимости $\alpha = 0.05$, то поправка Бонферрони требует использования скорректированного уровня:
$\alpha_{new}=\frac{0.05}{5}=0.01$

Теперь каждый отдельный тест должен иметь p-значение меньше 0.01, чтобы результат считался статистически значимым.

Реализация поправки Бонферрони в Python. Используем библиотеку statsmodels для применения коррекции Бонферрони к множественным 
p-значениям:

In [2]:
from statsmodels.stats.multitest import multipletests
import numpy as np

In [3]:
# Список p-значений от нескольких тестов
p_values = np.array([0.01, 0.03, 0.04, 0.06, 0.02])

# Применение поправки Бонферрони
corrected_pvals = multipletests(p_values, alpha=0.05, method='bonferroni')

# Вывод результатов
print("Скорректированные p-значения:", corrected_pvals[1])
print("Результаты значимости (True - отвергаем H0):", corrected_pvals[0])


Скорректированные p-значения: [0.05 0.15 0.2  0.3  0.1 ]
Результаты значимости (True - отвергаем H0): [ True False False False False]


# Достоинства и недостатки поправки Бонферрони
## Плюсы:

- Простота и легкость интерпретации.
- Защита от ложноположительных результатов (ошибок I рода).
- Подходит для небольшого количества тестов.

## Минусы:

- Консервативность — увеличивает вероятность ошибки второго рода (False Negative), так как сложнее выявить истинные эффекты.
- Непригоден для большого количества тестов, где другие методы могут быть более точными.

# Альтернативы поправке Бонферрони:
- Поправка Холма (Holm’s Correction):
Менее консервативна, но также контролирует общий уровень значимости.
- Поправка Бенджамини-Хохберга (Benjamini-Hochberg FDR):
Контролирует долю ложных открытий (False Discovery Rate, FDR), более подходит для больших данных.

In [4]:
corrected_pvals_bh = multipletests(p_values, alpha=0.05, method='fdr_bh')
corrected_pvals_h = multipletests(p_values, alpha=0.05, method='holm')

print("Скорректированные p-значения (FDR):", corrected_pvals_bh[1])
print("Скорректированные p-значения (Holm):", corrected_pvals_h[1])

Скорректированные p-значения (FDR): [0.05 0.05 0.05 0.06 0.05]
Скорректированные p-значения (Holm): [0.05 0.09 0.09 0.09 0.08]


# Заключение
- Используйте поправку Бонферрони, если:
    - Количество тестов невелико (например, < 10).
    - Требуется строгое управление ошибками первого рода.
- Используйте FDR-коррекцию, если:
    - Количество тестов велико (например, сотни или тысячи).
    - Нужен баланс между точностью и чувствительностью.

# Пример

In [5]:
p_values = np.array([0.001, 0.02, 0.03, 0.005, 0.15, 0.25, 0.04, 0.08, 0.12, 0.045])

corrected_pvals_b = multipletests(p_values, alpha=0.05, method='bonferroni')
corrected_pvals_h = multipletests(p_values, alpha=0.05, method='holm')
corrected_pvals_bh = multipletests(p_values, alpha=0.05, method='fdr_bh')

print("Скорректированные p-значения (Bonferroni):", np.round(corrected_pvals_b[1], 2))
print("Скорректированные p-значения (Holm):", np.round(corrected_pvals_h[1], 2))
print("Скорректированные p-значения (FDR):", np.round(corrected_pvals_bh[1], 2))


Скорректированные p-значения (Bonferroni): [0.01 0.2  0.3  0.05 1.   1.   0.4  0.8  1.   0.45]
Скорректированные p-значения (Holm): [0.01 0.16 0.21 0.04 0.36 0.36 0.24 0.32 0.36 0.24]
Скорректированные p-значения (FDR): [0.01 0.07 0.08 0.02 0.17 0.25 0.08 0.11 0.15 0.08]


In [6]:
len(p_values)

10

In [7]:
# Нахождение индексов выбросов
true_indices = [index for index, value in enumerate(corrected_pvals_b[1]) if value <= 0.05]
print("Индексы успешных тестов (Bonferroni):", true_indices)

true_indices = [index for index, value in enumerate(corrected_pvals_h[1]) if value <= 0.05]
print("Индексы успешных тестов (Holm):", true_indices)

true_indices = [index for index, value in enumerate(corrected_pvals_bh[1]) if value <= 0.05]
print("Индексы успешных тестов (FDR):", true_indices)

Индексы успешных тестов (Bonferroni): [0, 3]
Индексы успешных тестов (Holm): [0, 3]
Индексы успешных тестов (FDR): [0, 3]
